## Task Overview

This task counts only the vehicles moving **toward downtown** in two traffic videos. The approach combines motion-based detection with a lightweight tracking method so that cars can be counted reliably as they move through the scene.

The detection stage uses two motion cues: **MOG2 background subtraction** and **frame differencing**. These are applied inside a defined road region so that attention remains focused on the roadway rather than on unrelated background areas. After motion is detected, contour-based filtering removes unlikely shapes using constraints such as size, aspect ratio, and solidity. Nearby fragmented detections are then merged so that one car is less likely to appear as several separate blobs.

A custom tracker is used to maintain object identities across frames. Matching is performed first by overlap and then, when needed, by centroid distance. Tracks are only treated as reliable after they have persisted for enough frames. A separate gate region is placed on the right side of the road, representing the downtown direction. A car is counted only when it enters this gate, has moved with sufficient rightward velocity, and has remained stable long enough to avoid false counts.

The task is applied to both traffic videos. Each processed video is saved with the counted result drawn on the frames, and a final summary table reports the total cars counted toward downtown, the estimated cars per minute, and the duration of each video.

In [1]:
import os
import cv2
import numpy as np
from collections import OrderedDict

# ============================================================
# Exercise 1.2: Count cars moving toward downtown only
# ============================================================


# ------------------------------------------------------------
# Path setup
# ------------------------------------------------------------
INPUT_VIDEO_1 = os.path.join("..", "Exercise1_Files", "Traffic_Laramie_1.mp4")
INPUT_VIDEO_2 = os.path.join("..", "Exercise1_Files", "Traffic_Laramie_2.mp4")

for path in [INPUT_VIDEO_1, INPUT_VIDEO_2]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing video: {path}")



### Importing libraries and preparing input paths

This cell imports the main libraries required for video processing, numerical operations, and ordered track storage.

It then defines the paths to the two traffic videos used in the task. A file existence check is included for both paths so that execution stops immediately if either input video is missing. This avoids confusion later in the notebook when processing begins.

In [2]:
# ------------------------------------------------------------
# Geometry + box utilities
# ------------------------------------------------------------
def limit_box(x, y, w, h, frame_w, frame_h):
    x = max(0, min(int(x), frame_w - 1))
    y = max(0, min(int(y), frame_h - 1))
    w = max(1, min(int(w), frame_w - x))
    h = max(1, min(int(h), frame_h - y))
    return x, y, w, h


def box_midpoint(x, y, w, h):
    return int(x + w // 2), int(y + h // 2)


def inside_rectangle(point, rectangle):
    px, py = point
    rx, ry, rw, rh = rectangle
    return (rx <= px <= rx + rw) and (ry <= py <= ry + rh)


def overlap_ratio(box_a, box_b):
    ax, ay, aw, ah = box_a
    bx, by, bw, bh = box_b

    ax2, ay2 = ax + aw, ay + ah
    bx2, by2 = bx + bw, by + bh

    ix1, iy1 = max(ax, bx), max(ay, by)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)

    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = aw * ah + bw * bh - inter

    return inter / union if union > 0 else 0.0


def combine_nearby_boxes(candidate_boxes, spacing=55):
    """Merge nearby fragmented blobs that likely belong to one vehicle."""
    if not candidate_boxes:
        return []

    working = [list(b) for b in candidate_boxes]
    changed = True

    while changed:
        changed = False
        consumed = [False] * len(working)
        next_round = []

        for i in range(len(working)):
            if consumed[i]:
                continue

            x1, y1, w1, h1 = working[i]
            x2, y2 = x1 + w1, y1 + h1
            consumed[i] = True

            for j in range(i + 1, len(working)):
                if consumed[j]:
                    continue

                x3, y3, w3, h3 = working[j]
                x4, y4 = x3 + w3, y3 + h3

                separated = (
                    x4 < x1 - spacing or
                    x3 > x2 + spacing or
                    y4 < y1 - spacing or
                    y3 > y2 + spacing
                )

                if not separated:
                    consumed[j] = True
                    changed = True
                    x1 = min(x1, x3)
                    y1 = min(y1, y3)
                    x2 = max(x2, x4)
                    y2 = max(y2, y4)

            next_round.append([x1, y1, x2 - x1, y2 - y1])

        working = next_round

    return [tuple(map(int, b)) for b in working]



### Defining geometry and bounding-box helper functions

This cell defines the utility functions used throughout the counting pipeline.

`limit_box()` keeps a rectangle inside the valid frame boundaries, which is useful when cropping or drawing regions.  
`box_midpoint()` calculates the centre of a bounding box, which is later used for tracking and direction estimation.  
`inside_rectangle()` checks whether a point lies inside a given gate region.  
`overlap_ratio()` calculates Intersection over Union, which measures how strongly two boxes overlap.

`combine_nearby_boxes()` merges fragmented motion blobs that likely belong to the same vehicle. This is important because one moving car can sometimes appear as multiple separate foreground regions after motion segmentation.

In [3]:
# ------------------------------------------------------------
# Track manager
# ------------------------------------------------------------
class VehicleTrackManager:
    """
    Same tracking logic as the reference:
    - Greedy IoU matching first
    - Centroid-distance fallback second
    - Track confirmation using hit count
    """
    def __init__(self, max_lost=25, min_overlap=0.15, max_center_distance=140, confirm_hits=3):
        self.next_track_id = 1
        self.active_tracks = OrderedDict()
        self.max_lost = max_lost
        self.min_overlap = min_overlap
        self.max_center_distance = max_center_distance
        self.confirm_hits = confirm_hits

    def update_tracks(self, detections):
        for track_id in self.active_tracks:
            self.active_tracks[track_id]["was_updated"] = False

        if len(self.active_tracks) == 0:
            for det in detections:
                self._start_track(det)
            return

        existing_ids = list(self.active_tracks.keys())
        existing_boxes = [self.active_tracks[tid]["bbox"] for tid in existing_ids]

        matched_track_indices = set()
        matched_detection_indices = set()

        # Stage 1: greedy IoU matching
        scored_pairs = []
        for t_idx, track_box in enumerate(existing_boxes):
            for d_idx, det_box in enumerate(detections):
                scored_pairs.append((overlap_ratio(track_box, det_box), t_idx, d_idx))

        scored_pairs.sort(reverse=True, key=lambda item: item[0])

        for score, t_idx, d_idx in scored_pairs:
            if score < self.min_overlap:
                break
            if t_idx in matched_track_indices or d_idx in matched_detection_indices:
                continue

            track_id = existing_ids[t_idx]
            self._refresh_track(track_id, detections[d_idx])
            matched_track_indices.add(t_idx)
            matched_detection_indices.add(d_idx)

        # Stage 2: centroid fallback
        leftover_track_indices = [i for i in range(len(existing_ids)) if i not in matched_track_indices]
        leftover_detection_indices = [i for i in range(len(detections)) if i not in matched_detection_indices]

        for t_idx in leftover_track_indices:
            track_id = existing_ids[t_idx]
            x, y, w, h = self.active_tracks[track_id]["bbox"]
            track_center = box_midpoint(x, y, w, h)

            nearest_det_idx = None
            nearest_distance = float("inf")

            for d_idx in leftover_detection_indices:
                x2, y2, w2, h2 = detections[d_idx]
                det_center = box_midpoint(x2, y2, w2, h2)
                distance = np.hypot(track_center[0] - det_center[0], track_center[1] - det_center[1])

                if distance < nearest_distance:
                    nearest_distance = distance
                    nearest_det_idx = d_idx

            if nearest_det_idx is not None and nearest_distance <= self.max_center_distance:
                self._refresh_track(track_id, detections[nearest_det_idx])
                matched_detection_indices.add(nearest_det_idx)
                leftover_detection_indices = [idx for idx in leftover_detection_indices if idx != nearest_det_idx]

        # Mark unmatched tracks as missed
        expired_ids = []
        for track_id in list(self.active_tracks.keys()):
            if not self.active_tracks[track_id]["was_updated"]:
                self.active_tracks[track_id]["lost_frames"] += 1
            if self.active_tracks[track_id]["lost_frames"] > self.max_lost:
                expired_ids.append(track_id)

        for track_id in expired_ids:
            del self.active_tracks[track_id]

        # Start new tracks for unmatched detections
        for d_idx, det in enumerate(detections):
            if d_idx not in matched_detection_indices:
                self._start_track(det)

    def get_confirmed_tracks(self):
        return {
            track_id: track_data
            for track_id, track_data in self.active_tracks.items()
            if track_data["hit_count"] >= self.confirm_hits
        }

    def _start_track(self, bbox):
        track_id = self.next_track_id
        self.next_track_id += 1
        self.active_tracks[track_id] = {
            "bbox": bbox,
            "hit_count": 1,
            "lost_frames": 0,
            "was_updated": True,
        }

    def _refresh_track(self, track_id, bbox):
        self.active_tracks[track_id]["bbox"] = bbox
        self.active_tracks[track_id]["hit_count"] += 1
        self.active_tracks[track_id]["lost_frames"] = 0
        self.active_tracks[track_id]["was_updated"] = True



### Tracking vehicles across frames

This cell defines the custom track manager used to follow vehicles over time.

The tracker first tries to match detections to existing tracks using box overlap. Strong matches are handled first through a greedy IoU-based approach. When overlap is not enough, centroid distance is used as a fallback so that slightly shifted detections can still be linked to the correct object.

Each track stores its current box, the number of successful matches, the number of lost frames, and whether it was updated in the current step. Tracks that remain unmatched for too long are removed, while new detections start new tracks.

Only tracks with enough successful matches are treated as confirmed. This makes the later counting step more stable and helps suppress short-lived false detections.

In [4]:
# ------------------------------------------------------------
# Detection pipeline
# ------------------------------------------------------------
def detect_vehicle_boxes(
    frame,
    previous_roi_gray,
    background_model,
    road_region,
    min_area=2600,
    min_width=65,
    min_height=26,
    min_aspect=1.25,
    max_aspect=6.5,
    min_solidity=0.55,
    merge_spacing=55,
):
    """
    Same detection logic as the reference:
    - crop to ROI
    - MOG2 foreground mask
    - frame differencing
    - OR fuse
    - morphology
    - contour filtering
    - merge split vehicle blobs
    """
    frame_h, frame_w = frame.shape[:2]
    rx, ry, rw, rh = limit_box(*road_region, frame_w, frame_h)
    roi_frame = frame[ry:ry + rh, rx:rx + rw]

    gray_roi = cv2.cvtColor(roi_frame, cv2.COLOR_BGR2GRAY)
    blurred_gray_roi = cv2.GaussianBlur(gray_roi, (7, 7), 0)

    foreground_mask = background_model.apply(roi_frame)
    foreground_mask = cv2.medianBlur(foreground_mask, 5)
    _, foreground_mask = cv2.threshold(foreground_mask, 200, 255, cv2.THRESH_BINARY)

    if previous_roi_gray is None:
        previous_roi_gray = blurred_gray_roi.copy()

    frame_delta = cv2.absdiff(previous_roi_gray, blurred_gray_roi)
    _, difference_mask = cv2.threshold(frame_delta, 20, 255, cv2.THRESH_BINARY)

    fused_motion_mask = cv2.bitwise_or(foreground_mask, difference_mask)

    open_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17))
    fused_motion_mask = cv2.morphologyEx(fused_motion_mask, cv2.MORPH_OPEN, open_kernel, iterations=1)
    fused_motion_mask = cv2.morphologyEx(fused_motion_mask, cv2.MORPH_CLOSE, close_kernel, iterations=2)

    contours, _ = cv2.findContours(fused_motion_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    full_frame_boxes = []

    for contour in contours:
        contour_area = cv2.contourArea(contour)
        if contour_area < min_area:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        if w < min_width or h < min_height:
            continue

        aspect_ratio = w / float(h + 1e-9)
        if not (min_aspect <= aspect_ratio <= max_aspect):
            continue

        hull = cv2.convexHull(contour)
        hull_area = cv2.contourArea(hull) + 1e-9
        solidity = contour_area / hull_area
        if solidity < min_solidity:
            continue

        full_frame_boxes.append((x + rx, y + ry, w, h))

    full_frame_boxes = combine_nearby_boxes(full_frame_boxes, spacing=merge_spacing)
    return full_frame_boxes, blurred_gray_roi



### Detecting vehicle boxes inside the road region

This cell defines the motion-based detection stage.

The frame is first cropped to the road region so that only the relevant traffic area is processed. Inside that region, the image is converted to grayscale and blurred to reduce noise.

Two motion cues are then extracted:

- **Background subtraction**, which captures persistent moving foreground regions.
- **Frame differencing**, which detects short-term motion between consecutive frames.

These two masks are combined, then cleaned with morphological opening and closing. External contours are extracted from the final mask, and each contour is filtered using area, width, height, aspect ratio, and solidity. These checks are designed to keep vehicle-like regions and reject many irrelevant blobs.

The detected boxes are finally merged when nearby fragments likely belong to the same vehicle.

In [5]:
# ------------------------------------------------------------
# Main counting routine
# ------------------------------------------------------------
def process_downtown_count(video_path, output_path):
    capture = cv2.VideoCapture(video_path)
    if not capture.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    frame_w = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = capture.get(cv2.CAP_PROP_FPS) or 25.0

    # Same ROI and gate logic as the reference
    road_region = (0, int(frame_h * 0.38), frame_w, int(frame_h * 0.62))
    downtown_gate = (int(frame_w * 0.62), int(frame_h * 0.38), int(frame_w * 0.38), int(frame_h * 0.45))

    writer = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (frame_w, frame_h)
    )

    background_model = cv2.createBackgroundSubtractorMOG2(
        history=700,
        varThreshold=35,
        detectShadows=False
    )

    track_manager = VehicleTrackManager(
        max_lost=25,
        min_overlap=0.15,
        max_center_distance=140,
        confirm_hits=3
    )

    previous_roi_gray = None
    previous_centers = {}   # track_id -> (cx, cy)
    smoothed_velocity = {}  # track_id -> (vx, vy)
    already_counted = set()
    downtown_count = 0

    warmup_target = 25
    warmup_seen = 0

    def estimate_velocity(track_id, center_point, smoothing=0.6):
        if track_id not in previous_centers:
            smoothed_velocity[track_id] = (0.0, 0.0)
            return 0.0, 0.0

        old_center = previous_centers[track_id]
        instant_vx = center_point[0] - old_center[0]
        instant_vy = center_point[1] - old_center[1]

        prev_vx, prev_vy = smoothed_velocity.get(track_id, (0.0, 0.0))
        filt_vx = smoothing * instant_vx + (1 - smoothing) * prev_vx
        filt_vy = smoothing * instant_vy + (1 - smoothing) * prev_vy
        smoothed_velocity[track_id] = (filt_vx, filt_vy)
        return filt_vx, filt_vy

    while True:
        ok, frame = capture.read()
        if not ok:
            break

        rx, ry, rw, rh = limit_box(*road_region, frame_w, frame_h)
        roi_crop = frame[ry:ry + rh, rx:rx + rw]

        if warmup_seen < warmup_target:
            background_model.apply(roi_crop)
            warmup_seen += 1
            writer.write(frame)
            continue

        detections, previous_roi_gray = detect_vehicle_boxes(
            frame,
            previous_roi_gray,
            background_model,
            road_region
        )

        track_manager.update_tracks(detections)
        confirmed_tracks = track_manager.get_confirmed_tracks()

        cv2.rectangle(frame, (rx, ry), (rx + rw, ry + rh), (255, 255, 0), 2)

        gx, gy, gw, gh = downtown_gate
        cv2.rectangle(frame, (gx, gy), (gx + gw, gy + gh), (0, 255, 255), 3)

        for track_id, track_info in confirmed_tracks.items():
            x, y, w, h = track_info["bbox"]
            center = box_midpoint(x, y, w, h)

            vx, vy = estimate_velocity(track_id, center, smoothing=0.6)

            moving_toward_downtown = (vx > 1.2)
            persistent_track = (track_info["hit_count"] >= 8)
            is_inside_gate = inside_rectangle(center, downtown_gate)

            if is_inside_gate and moving_toward_downtown and persistent_track and track_id not in already_counted:
                already_counted.add(track_id)
                downtown_count += 1

            previous_centers[track_id] = center

            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        cv2.putText(
            frame,
            f"Downtown cars: {downtown_count}",
            (15, 55),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.3,
            (255, 255, 255),
            3
        )

        cv2.putText(
            frame,
            f"dets={len(detections)} tracks={len(confirmed_tracks)} counted={len(already_counted)}",
            (15, 100),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (255, 255, 255),
            2
        )

        writer.write(frame)

    capture.release()
    writer.release()
    return downtown_count

### Counting downtown-moving cars in one video

This cell defines the main routine that processes one video and counts cars moving toward downtown.

The video is opened first, and its dimensions and frame rate are read. A road region is then defined for detection, while a separate gate region on the right side of the road is used as the downtown counting area.

A background model and track manager are created, and a short warm-up phase is used so that the background subtractor can stabilise before counting begins.

For each frame after warm-up, vehicle detections are generated and passed to the tracker. Only confirmed tracks are considered for counting. A smoothed velocity estimate is calculated from the change in centroid position across frames. A vehicle is counted only when all conditions are satisfied:

- the centroid is inside the downtown gate,
- the horizontal velocity is positive enough,
- the track has persisted for enough frames,
- and the same track has not already been counted.

The frame is annotated with the road region, downtown gate, vehicle boxes, and running counters, then written to the output video.

In [6]:
# ------------------------------------------------------------
# Run both videos
# ------------------------------------------------------------
OUTPUT_VIDEO_1 = "exercise1_task2_Traffic_Laramie_1_FIXED.mp4"
OUTPUT_VIDEO_2 = "exercise1_task2_Traffic_Laramie_2_FIXED.mp4"

count_video_1 = process_downtown_count(INPUT_VIDEO_1, OUTPUT_VIDEO_1)
count_video_2 = process_downtown_count(INPUT_VIDEO_2, OUTPUT_VIDEO_2)

print("Traffic_Laramie_1 -> Downtown cars:", count_video_1, "Output:", OUTPUT_VIDEO_1)
print("Traffic_Laramie_2 -> Downtown cars:", count_video_2, "Output:", OUTPUT_VIDEO_2)


Traffic_Laramie_1 -> Downtown cars: 10 Output: exercise1_task2_Traffic_Laramie_1_FIXED.mp4
Traffic_Laramie_2 -> Downtown cars: 4 Output: exercise1_task2_Traffic_Laramie_2_FIXED.mp4


### Running the counting process on both traffic videos

This cell applies the downtown counting routine to both input videos.

Two output filenames are defined first. Each video is then processed independently, producing a counted result and a saved annotated video. The printed lines at the end report the final downtown vehicle count for each source video together with the corresponding output filename.

In [7]:
summary_rows = []

for video_path, output_path in [
    (INPUT_VIDEO_1, OUTPUT_VIDEO_1),
    (INPUT_VIDEO_2, OUTPUT_VIDEO_2),
]:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0:
        fps = 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    total_cars = process_downtown_count(video_path, output_path)

    duration_seconds = total_frames / fps
    duration_minutes = duration_seconds / 60.0 if duration_seconds > 0 else 1e-9
    cars_per_minute = total_cars / duration_minutes

    summary_rows.append((
        os.path.basename(video_path),
        total_cars,
        cars_per_minute,
        duration_minutes,
        output_path
    ))

print("\n=== Summary Table ===")
print("Video\t\t\tTotal cars toward downtown\tCars/min\tDuration(min)")
for name, total_cars, cars_per_minute, duration_minutes, output_path in summary_rows:
    print(f"{name}\t{total_cars}\t\t\t\t{cars_per_minute:.2f}\t\t{duration_minutes:.2f}")

print("\nSaved output videos:")
for _, _, _, _, output_path in summary_rows:
    print(output_path)


=== Summary Table ===
Video			Total cars toward downtown	Cars/min	Duration(min)
Traffic_Laramie_1.mp4	10				3.37		2.97
Traffic_Laramie_2.mp4	4				2.27		1.76

Saved output videos:
exercise1_task2_Traffic_Laramie_1_FIXED.mp4
exercise1_task2_Traffic_Laramie_2_FIXED.mp4


### Building the summary table

This final cell gathers summary statistics for both processed videos.

For each input file, the frame rate and total number of frames are read so that the full duration can be calculated. The counting routine is then applied again to obtain the total downtown vehicle count. Using the total duration in minutes, the cars-per-minute rate is calculated.

All values are stored in `summary_rows` and then printed in a compact summary table. The output video filenames are also listed at the end so that the saved results can be located easily.